In [3]:
# Gerekli paket kontrolü
import importlib.util

required_packages = ['pandas', 'numpy', 'openpyxl', 'plotly', 'gurobipy']
missing_packages = [
    package for package in required_packages
    if importlib.util.find_spec(package) is None
]

if missing_packages:
    print('Eksik paketler:', missing_packages)
    print('Anaconda Prompt içinde şu komutu çalıştırın:')
    print('pip install ' + ' '.join(missing_packages))
    raise ModuleNotFoundError('Önce eksik paketleri kurun, sonra notebooku yeniden başlatın.')

print('Gerekli Python paketleri hazır.')


# ============================================================
# CNC OVERTIME MINIMIZATION EXACT MILP (SMALL / SIMPLE TEST DATASET)
# Gurobi + Integer Job Splitting + Global Machine No-Overlap + HTML Gantt
# ============================================================
# REVİZYONLAR (hoca geri bildirimine göre):
#   1. BIG_M: 10**7 -> 10,000
#   2. Gurobi log dosyasi eklendi: gurobi_exact_small.log
#   3. Model size raporlamaya n_binary/n_integer/n_continuous eklendi.
#   4. SolCount=0 durumu artik RuntimeError firlatmiyor.
#   5. Objective sadelestirildi (sadece total_overtime).
# ============================================================

import os
import math
import re
from dataclasses import dataclass
from datetime import datetime, timedelta, time
from collections import defaultdict
from pathlib import Path

import pandas as pd
import numpy as np

from gurobipy import Model, GRB, quicksum

import plotly.graph_objects as go

from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter


BASE_DIR = Path(r'C:\Users\emrec\Documents\parallel_scheduling\exactkucuksplit')


def find_input_file(possible_names, file_description):
    for filename in possible_names:
        candidate = BASE_DIR / filename
        if candidate.is_file():
            print(f"{file_description:<22}: {candidate}")
            return candidate.resolve()

    available = "\n".join([p.name for p in sorted(BASE_DIR.glob('*.xlsx'))])
    raise FileNotFoundError(
        f"\n{file_description} bulunamadi.\n"
        f"Aranan dosya adlari: {possible_names}\n"
        f"BASE_DIR: {BASE_DIR}\n"
        f"Mevcut Excel dosyalari:\n{available}"
    )

SHIPMENT_FILE = find_input_file(
    [
        "simple_test_sevkiyat.xlsx",
        "simple_test_sevkiyat(1).xlsx",
        "simple_test_sevkiyat(2).xlsx",
    ],
    "Shipment file",
)

SDST_FILE = find_input_file(
    [
        "SDST.xlsx",
        "SDST(1).xlsx",
        "SDST(5).xlsx",
    ],
    "SDST file",
)

MACHINE_GROUP_FILE = find_input_file(
    [
        "machine_group_data_simple.xlsx",
        "machine_group_data_simple(1).xlsx",
        "machine_group_data_simple(2).xlsx",
    ],
    "Machine-group file",
)

if not BASE_DIR.exists():
    BASE_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_SCHEDULE_FILE   = BASE_DIR / "optimized_exact_gurobi_schedule_split.xlsx"
OUTPUT_VALIDATION_FILE = BASE_DIR / "verification_validation_exact_gurobi_split.xlsx"
OUTPUT_GANTT_FILE      = BASE_DIR / "exact_gurobi_gantt_split.html"
GUROBI_LOG_FILE        = BASE_DIR / "gurobi_exact_small.log"

print("\nUsing files:")
print("BASE_DIR           =", BASE_DIR)
print("SHIPMENT_FILE      =", SHIPMENT_FILE)
print("SDST_FILE          =", SDST_FILE)
print("MACHINE_GROUP_FILE =", MACHINE_GROUP_FILE)
print("GUROBI_LOG_FILE    =", GUROBI_LOG_FILE)
print("\nOutputs will be saved in:", BASE_DIR)

PLANNING_START_HOUR = 7
DUE_TIME_HOUR       = 17

OVERTIME_START = time(17, 0)
OVERTIME_END   = time(21, 0)

DISALLOW_START_IN_OVERTIME = False

TARDINESS_LIMIT_DAYS = 1.0
TARDINESS_LIMIT_MIN  = TARDINESS_LIMIT_DAYS * 24 * 60

ALLOW_JOB_SPLITTING = True
MAX_SPLITS    = 2
MIN_SPLIT_QTY = 50

INITIAL_SETUP_MIN           = 10.0
SAME_DIAM_SETUP_MIN         = 5.0
DEFAULT_DIFF_DIAM_SETUP_MIN = 20.0

MIP_GAP        = 0.05
TIME_LIMIT_SEC = 3600
OUTPUT_FLAG    = 1

BIG_M = 10_000

EPS = 1e-5


def normalize_machine_name(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    s = s.replace(' ', '')
    s = s.replace(',', '.')
    s = s.replace('O', '0')
    s = re.sub(r'^T3\.', 'T.3.', s)
    if re.match(r'^T3\d+', s):
        s = 'T.3.' + s[2:]
    return s


def excel_serial_to_datetime_date(x):
    if pd.isna(x):
        return None
    if isinstance(x, datetime):
        return x.date()
    if isinstance(x, (int, float, np.integer, np.floating)):
        return (datetime(1899, 12, 30) + timedelta(days=float(x))).date()
    parsed = pd.to_datetime(x, errors='coerce')
    if pd.isna(parsed):
        return None
    return parsed.date()


def excel_time_to_timedelta(x):
    if pd.isna(x):
        return None
    if isinstance(x, timedelta):
        return x
    if isinstance(x, datetime):
        return timedelta(hours=x.hour, minutes=x.minute, seconds=x.second)
    if isinstance(x, time):
        return timedelta(hours=x.hour, minutes=x.minute, seconds=x.second)
    if isinstance(x, (int, float, np.integer, np.floating)):
        return timedelta(days=float(x))
    parsed = pd.to_datetime(str(x), errors='coerce')
    if pd.isna(parsed):
        return None
    return timedelta(hours=parsed.hour, minutes=parsed.minute, seconds=parsed.second)


def combine_excel_date_time(date_value, time_value):
    d  = excel_serial_to_datetime_date(date_value)
    td = excel_time_to_timedelta(time_value)
    if d is None or td is None:
        return None
    return datetime.combine(d, time(0, 0)) + td


def minutes_between(start_dt, end_dt):
    if end_dt <= start_dt:
        end_dt = end_dt + timedelta(days=1)
    return (end_dt - start_dt).total_seconds() / 60.0


def overlap_minutes(a_start, a_end, b_start, b_end):
    latest_start = max(a_start, b_start)
    earliest_end = min(a_end,   b_end)
    return max(0.0, (earliest_end - latest_start).total_seconds() / 60.0)


def overtime_overlap_minutes(start_dt, end_dt):
    """
    DUZELTME: Orijinal sevkiyat verisindeki 'Fazla mesai' sutunu ile
    tutarli olacak sekilde guncellendi.

    Eski mantik: sadece 17:00-21:00 penceresiyle KESISIM hesaplaniyordu.
    Bu yuzden 21:00'dan sonra biten isler icin (orn. 15:11-21:51) sadece
    17:00-21:00 = 240 dk sayiliyordu, 21:00-21:51 arasi (51 dk) kayboluyordu.
    Toplam baseline boylece 377 dk yerine 326 dk cikiyordu (51 dk fark).

    Yeni mantik: orijinal sevkiyat dosyasindaki gibi, 17:00'dan SONRA
    baslayan veya 17:00'i GECEN her isin, 17:00'dan itibaren BITENE KADAR
    tum suresi fazla mesai sayilir (ust sinir yok). Bu sekilde
    15:11-21:51 -> 17:00-21:51 arasi tam 291 dk olarak hesaplanir,
    326 + 51 = 377 dk baseline ile birebir eslesir.
    """
    if end_dt <= start_dt:
        return 0.0
    total       = 0.0
    current_day = start_dt.date()
    last_day    = end_dt.date()
    while current_day <= last_day:
        ot_start = datetime.combine(current_day, OVERTIME_START)
        # UST SINIR YOK: gun sonuna kadar (bir sonraki gunun 00:00'i) overtime sayilir
        ot_end   = datetime.combine(current_day + timedelta(days=1), time(0, 0))
        total   += overlap_minutes(start_dt, end_dt, ot_start, ot_end)
        current_day = current_day + timedelta(days=1)
    return total


def parse_overtime_text(value):
    if pd.isna(value):
        return 0.0
    s = str(value).strip().lower()
    if not s:
        return 0.0
    hours   = 0
    minutes = 0
    h = re.search(r'(\d+)\s*saat', s)
    m = re.search(r'(\d+)\s*dk',   s)
    if h:
        hours   = int(h.group(1))
    if m:
        minutes = int(m.group(1))
    if not h and not m:
        nums = re.findall(r'\d+', s)
        if nums:
            minutes = int(nums[0])
    return float(hours * 60 + minutes)


def mode_or_first(series, default=None):
    s = series.dropna()
    if len(s) == 0:
        return default
    mo = s.mode()
    if len(mo) > 0:
        return mo.iloc[0]
    return s.iloc[0]


def dt_to_min(dt, planning_start):
    return (dt - planning_start).total_seconds() / 60.0


def min_to_dt(minutes, planning_start):
    return planning_start + timedelta(minutes=float(minutes))


@dataclass
class Job:
    job_id:               str
    part_no:              str
    due_date:             datetime
    quantity:             int
    diameter:             float
    eligible_groups_op10: list
    eligible_groups_op20: list


def load_shipment_operations(path):
    raw = pd.read_excel(path, sheet_name=0, header=5, usecols='D:N', engine='openpyxl')
    raw.columns = [str(c).strip().replace(':', '') for c in raw.columns]
    raw = raw.dropna(how='all')

    rename_map = {
        'Tarih':                       'date',
        'Başlangıç Saat':              'start_time',
        'Bitiş Saat':                  'finish_time',
        'Parça No':                    'part_no',
        'Makine No':                   'machine',
        'CNC-1 operasyonu(piston)':    'op10_flag',
        'CNC-2 operasyonu (saplama)':  'op20_flag',
        'Adet':                        'quantity',
        'Çap':                         'diameter',
        'Makine Grubu':                'machine_group',
        'Fazla mesai':                 'overtime_text',
    }
    raw = raw.rename(columns=rename_map)

    required = ['date','start_time','finish_time','part_no','machine','quantity','diameter','machine_group']
    for col in required:
        if col not in raw.columns:
            raise ValueError(f'Missing column in shipment sheet: {col}')

    raw = raw[raw['part_no'].notna()].copy()
    raw['machine']       = raw['machine'].apply(normalize_machine_name)
    raw['part_no']       = raw['part_no'].astype(int).astype(str)
    raw['quantity']      = pd.to_numeric(raw['quantity'],      errors='coerce').fillna(0).astype(int)
    raw['diameter']      = pd.to_numeric(raw['diameter'],      errors='coerce')
    raw['machine_group'] = pd.to_numeric(raw['machine_group'], errors='coerce').astype('Int64')

    raw['operation'] = np.where(
        raw.get('op10_flag').notna(), 10,
        np.where(raw.get('op20_flag').notna(), 20, np.nan)
    )
    raw = raw[raw['operation'].notna()].copy()
    raw['operation'] = raw['operation'].astype(int)

    raw['start_dt']     = [combine_excel_date_time(d, t) for d, t in zip(raw['date'], raw['start_time'])]
    raw['finish_dt']    = [combine_excel_date_time(d, t) for d, t in zip(raw['date'], raw['finish_time'])]
    raw['duration_min'] = [minutes_between(s, f) for s, f in zip(raw['start_dt'], raw['finish_dt'])]
    raw['unit_min']     = raw['duration_min'] / raw['quantity'].replace(0, np.nan)
    return raw


def load_orders_from_sayfa2(path, use_shipped_quantity=False):
    df = pd.read_excel(path, sheet_name=1, header=None, engine='openpyxl')
    jobs         = []
    current_date = None
    in_block     = False

    for _, row in df.iterrows():
        first_values = row.dropna().tolist()
        if len(first_values) == 0:
            continue

        possible_date = None
        for val in row.tolist():
            if pd.isna(val):
                continue
            if isinstance(val, (int, float, np.integer, np.floating)) and 40000 < float(val) < 60000:
                possible_date = excel_serial_to_datetime_date(val)
                break
            if isinstance(val, datetime):
                possible_date = val.date()
                break

        if possible_date is not None:
            current_date = possible_date
            in_block     = False
            continue

        row_text = ' '.join([str(v).lower() for v in row.dropna().tolist()])
        if 'sipariş' in row_text and 'adet' in row_text:
            in_block = True
            continue

        if in_block and current_date is not None:
            part        = row.iloc[4] if len(row) > 4 else None
            order_qty   = row.iloc[5] if len(row) > 5 else None
            shipped_qty = row.iloc[6] if len(row) > 6 else None

            if pd.isna(part) or pd.isna(order_qty):
                continue
            try:
                part_no = str(int(part))
            except Exception:
                continue

            qty_source = shipped_qty if use_shipped_quantity and not pd.isna(shipped_qty) else order_qty
            qty = int(round(float(qty_source)))
            if qty <= 0:
                continue

            due_dt = datetime.combine(current_date, time(DUE_TIME_HOUR, 0))
            job_id = f'{current_date.isoformat()}__{part_no}'
            jobs.append({'job_id': job_id, 'part_no': part_no, 'due_date': due_dt, 'quantity': qty})

    orders = pd.DataFrame(jobs)
    if orders.empty:
        raise ValueError('Sayfa2 uzerinde siparis satiri bulunamadi.')

    before_n = len(orders)
    orders = (
        orders.groupby(['job_id','part_no'], as_index=False)
        .agg(quantity=('quantity','sum'), due_date=('due_date','min'))
    )
    after_n = len(orders)
    if before_n != after_n:
        print(f'Duplicate order rows aggregated: {before_n} -> {after_n} unique jobs')

    return orders


def load_machine_groups(path):
    mg = pd.read_excel(path, sheet_name=0, engine='openpyxl')
    mg.columns = [str(c).strip() for c in mg.columns]
    mg = mg.dropna(subset=['Machine_number','Group']).copy()
    mg['Machine_number'] = mg['Machine_number'].apply(normalize_machine_name)
    mg['Group']          = pd.to_numeric(mg['Group'], errors='coerce').astype(int)

    group_to_machines = defaultdict(list)
    machine_to_group  = {}
    for _, r in mg.iterrows():
        m = r['Machine_number']
        g = int(r['Group'])
        group_to_machines[g].append(m)
        machine_to_group[m] = g
    return dict(group_to_machines), machine_to_group


def load_sdst(path):
    sdst = pd.read_excel(path, sheet_name=0, engine='openpyxl')
    sdst.columns = [str(c).strip() for c in sdst.columns]
    setup = {}
    for _, r in sdst.dropna(subset=['diam_from','to_diam','setup_time']).iterrows():
        d1 = float(r['diam_from'])
        d2 = float(r['to_diam'])
        setup[(d1, d2)] = float(r['setup_time'])
    return setup


def build_problem_data(shipment_file, sdst_file, machine_group_file,
                       use_shipped_quantity=False):
    ops    = load_shipment_operations(shipment_file)
    orders = load_orders_from_sayfa2(shipment_file, use_shipped_quantity=use_shipped_quantity)
    group_to_machines, machine_to_group = load_machine_groups(machine_group_file)
    setup_dict = load_sdst(sdst_file)

    part_diam = (ops.groupby('part_no')['diameter']
                    .agg(lambda x: mode_or_first(x, default=0))
                    .to_dict())

    eligible = defaultdict(lambda: defaultdict(set))
    for _, r in ops.iterrows():
        eligible[r['part_no']][int(r['operation'])].add(int(r['machine_group']))

    unit_time = {}
    grouped   = ops.dropna(subset=['unit_min','machine_group']).groupby(
                    ['part_no','operation','machine_group'])
    for key, g in grouped:
        unit_time[(str(key[0]), int(key[1]), int(key[2]))] = float(g['unit_min'].median())

    fallback_part_op = (ops.dropna(subset=['unit_min'])
                           .groupby(['part_no','operation'])['unit_min']
                           .median().to_dict())
    fallback_op  = (ops.dropna(subset=['unit_min'])
                       .groupby(['operation'])['unit_min']
                       .median().to_dict())
    global_unit  = float(ops['unit_min'].dropna().median())

    jobs       = []
    all_groups = sorted(group_to_machines.keys())

    for _, r in orders.iterrows():
        part_no = str(r['part_no'])
        q       = int(r['quantity'])
        diam    = float(part_diam.get(part_no, ops['diameter'].dropna().median()))

        eg10 = sorted(list(eligible[part_no].get(10, set(all_groups))))
        eg20 = sorted(list(eligible[part_no].get(20, set(eg10 if eg10 else all_groups))))
        if not eg10:
            eg10 = all_groups
        if not eg20:
            eg20 = eg10

        jobs.append(Job(
            job_id=r['job_id'],
            part_no=part_no,
            due_date=r['due_date'],
            quantity=q,
            diameter=diam,
            eligible_groups_op10=eg10,
            eligible_groups_op20=eg20,
        ))

    # DUZELTME: artik overtime_overlap_minutes() ile birebir ayni mantigi
    # kullaniyor (ust sinir yok, 17:00'dan sonraki tum sure sayilir).
    # Eskiden burada AYRI ve eski (21:00 ust sinirli) mantik vardi, bu da
    # baseline'in 377 yerine 326 cikmasina sebep oluyordu.
    ops['shipment_file_overtime_min'] = ops.apply(
        lambda row: overtime_overlap_minutes(row['start_dt'], row['finish_dt'])
                    if row['start_dt'] is not None and row['finish_dt'] is not None else 0.0,
        axis=1
    )
    baseline_overtime_min = float(ops['shipment_file_overtime_min'].sum())

    planning_start = datetime.combine(
        min([j.due_date.date() for j in jobs]),
        time(PLANNING_START_HOUR, 0)
    )

    return {
        'ops':                   ops,
        'baseline_overtime_min': baseline_overtime_min,
        'orders':                orders,
        'jobs':                  jobs,
        'group_to_machines':     group_to_machines,
        'machine_to_group':      machine_to_group,
        'setup_dict':            setup_dict,
        'unit_time':             unit_time,
        'fallback_part_op':      fallback_part_op,
        'fallback_op':           fallback_op,
        'global_unit':           global_unit,
        'planning_start':        planning_start,
    }


def get_unit_time(data, part_no, operation, group):
    key = (str(part_no), int(operation), int(group))
    if key in data['unit_time']:
        return data['unit_time'][key]
    key2 = (str(part_no), int(operation))
    if key2 in data['fallback_part_op']:
        return float(data['fallback_part_op'][key2])
    if int(operation) in data['fallback_op']:
        return float(data['fallback_op'][int(operation)])
    return data['global_unit']


def get_setup_time(data, from_diam, to_diam):
    if from_diam is None:
        return INITIAL_SETUP_MIN
    d1 = float(from_diam)
    d2 = float(to_diam)
    if abs(d1 - d2) < 1e-9:
        return SAME_DIAM_SETUP_MIN
    if (d1, d2) in data['setup_dict']:
        return data['setup_dict'][(d1, d2)]
    return DEFAULT_DIFF_DIAM_SETUP_MIN


def common_eligible_groups(job):
    common = sorted(list(
        set(job.eligible_groups_op10).intersection(set(job.eligible_groups_op20))
    ))
    if common:
        return common
    return sorted(job.eligible_groups_op10)


def build_overtime_windows(planning_start, latest_due):
    """
    DUZELTME: Pencere artik 17:00'dan GECE YARISINA kadar uzaniyor
    (eskiden 17:00-21:00 idi). Bu, overtime_overlap_minutes() fonksiyonunun
    yeni mantigiyla (ust sinir yok, sadece 17:00'dan sonrasi) BIREBIR
    TUTARLI olmasi icin gerekli.

    Eskiden bu iki fonksiyon FARKLI pencereler kullaniyordu:
      - build_overtime_windows(): Gurobi'nin optimize ettigi ot_vars icin
        17:00-21:00 penceresi kuruyordu.
      - overtime_overlap_minutes(): rapor/extract asamasinda 17:00-24:00
        (sinirsiz) hesapliyordu.
    Bu tutarsizlik yuzunden Gurobi'nin Objective=0.0 dedigi cozumun
    rapordaki gercek toplami 400+ dk cikiyordu (iki farkli tanim).
    Artik ikisi de ayni pencereyi (17:00-24:00) kullaniyor.
    """
    horizon_end = (latest_due
                   + timedelta(minutes=TARDINESS_LIMIT_MIN)
                   + timedelta(days=1))
    windows     = []
    current_day = planning_start.date()
    while datetime.combine(current_day, time(0, 0)) <= horizon_end:
        w_start = datetime.combine(current_day, OVERTIME_START)
        w_end   = datetime.combine(current_day + timedelta(days=1), time(0, 0))  # gece yarisina kadar
        u = dt_to_min(w_end,   planning_start)
        l = dt_to_min(w_start, planning_start)
        if u > 0:
            windows.append((max(0.0, l), u, current_day.isoformat()))
        current_day = current_day + timedelta(days=1)
    return windows


def solve_exact_gurobi(data):
    jobs = data['jobs']

    job_ids = [j.job_id for j in jobs]
    if len(job_ids) != len(set(job_ids)):
        vc         = pd.Series(job_ids).value_counts()
        duplicates = vc[vc > 1].index.tolist()
        raise ValueError(f'Duplicate job_id: {duplicates[:20]}')

    machines         = sorted(data['machine_to_group'].keys())
    groups           = sorted(data['group_to_machines'].keys())
    ops_list         = [10, 20]
    planning_start   = data['planning_start']
    latest_due       = max(j.due_date for j in jobs)
    overtime_windows = build_overtime_windows(planning_start, latest_due)

    max_splits = MAX_SPLITS if ALLOW_JOB_SPLITTING else 1
    split_ids  = list(range(1, max_splits + 1))

    tasks = []
    for j in jobs:
        for r in split_ids:
            for op in ops_list:
                tasks.append((j.job_id, r, op))

    job_by_id      = {j.job_id: j for j in jobs}
    allowed_groups = {j.job_id: common_eligible_groups(j) for j in jobs}

    eligible_machines_by_task = {}
    for j in jobs:
        for r in split_ids:
            for op in ops_list:
                ms = []
                for g in allowed_groups[j.job_id]:
                    ms.extend(data['group_to_machines'].get(g, []))
                eligible_machines_by_task[(j.job_id, r, op)] = sorted(set(ms))

    horizon_min = dt_to_min(
        latest_due + timedelta(minutes=TARDINESS_LIMIT_MIN) + timedelta(days=1),
        planning_start
    )
    print(f'\nSchedule horizon : {horizon_min:.0f} min  ({horizon_min/60:.1f} h)')
    print(f'Big-M used       : {BIG_M}  (revised from 10**7 = {10**7}; {10**7//BIG_M}x tighter)')

    M = BIG_M

    model = Model('Exact_Gurobi_Split_GlobalMachineNoOverlap_Small')

    model.Params.LogFile = str(GUROBI_LOG_FILE)

    model.Params.MIPGap     = MIP_GAP
    model.Params.OutputFlag = OUTPUT_FLAG
    if TIME_LIMIT_SEC is not None:
        model.Params.TimeLimit = TIME_LIMIT_SEC

    active = model.addVars(
        [(j.job_id, r) for j in jobs for r in split_ids],
        vtype=GRB.BINARY, name='active'
    )
    qty = model.addVars(
        [(j.job_id, r) for j in jobs for r in split_ids],
        vtype=GRB.INTEGER, lb=0, name='qty'
    )
    z_group = model.addVars(
        [(j.job_id, r, g) for j in jobs for r in split_ids
                           for g in allowed_groups[j.job_id]],
        vtype=GRB.BINARY, name='z_group'
    )
    qty_group = model.addVars(
        [(j.job_id, r, g) for j in jobs for r in split_ids
                           for g in allowed_groups[j.job_id]],
        vtype=GRB.INTEGER, lb=0, name='qty_group'
    )
    x = model.addVars(
        [(t[0], t[1], t[2], m)
         for t in tasks for m in eligible_machines_by_task[t]],
        vtype=GRB.BINARY, name='x'
    )

    S         = model.addVars(tasks, lb=0.0, name='S')
    C         = model.addVars(tasks, lb=0.0, name='C')
    proc_time = model.addVars(tasks, lb=0.0, name='proc_time')

    arc_keys = []
    for ti in tasks:
        for tj in tasks:
            if ti == tj:
                continue
            common_ms = set(eligible_machines_by_task[ti]).intersection(
                         eligible_machines_by_task[tj])
            for m in common_ms:
                arc_keys.append((ti[0],ti[1],ti[2], tj[0],tj[1],tj[2], m))
    arc = model.addVars(arc_keys, vtype=GRB.BINARY, name='arc')

    first = model.addVars(
        [(t[0],t[1],t[2],m) for t in tasks for m in eligible_machines_by_task[t]],
        vtype=GRB.BINARY, name='first'
    )
    last = model.addVars(
        [(t[0],t[1],t[2],m) for t in tasks for m in eligible_machines_by_task[t]],
        vtype=GRB.BINARY, name='last'
    )

    job_finish = model.addVars([j.job_id for j in jobs], lb=0.0, name='job_finish')
    makespan   = model.addVar(lb=0.0, name='makespan')

    ot_vars = {}
    for t in tasks:
        for w_idx, (_l, _u, _nm) in enumerate(overtime_windows):
            ot_vars[(t, w_idx)] = model.addVar(
                lb=0.0, name=f'ot_{t[0]}{t[1]}{t[2]}_{w_idx}')

    start_side = {}
    if DISALLOW_START_IN_OVERTIME:
        for t in tasks:
            for w_idx, _ in enumerate(overtime_windows):
                start_side[(t, w_idx)] = model.addVar(
                    vtype=GRB.BINARY,
                    name=f'start_side_{t[0]}{t[1]}{t[2]}_{w_idx}'
                )

    for j in jobs:
        jid   = j.job_id
        Q     = int(j.quantity)
        min_q = min(MIN_SPLIT_QTY, Q)

        model.addConstr(active[jid, 1] == 1, name=f'first_split_active_{jid}')
        model.addConstr(quicksum(qty[jid, r] for r in split_ids) == Q,
                        name=f'qty_conservation_{jid}')

        for r in split_ids:
            if r > 1:
                model.addConstr(active[jid, r] <= active[jid, r-1],
                                name=f'split_order_{jid}_{r}')
                if Q < 2 * MIN_SPLIT_QTY:
                    model.addConstr(active[jid, r] == 0,
                                    name=f'no_small_split_{jid}_{r}')

            model.addConstr(qty[jid, r] <= Q * active[jid, r],
                            name=f'qty_ub_{jid}_{r}')
            model.addConstr(qty[jid, r] >= min_q * active[jid, r],
                            name=f'qty_lb_{jid}_{r}')
            model.addConstr(
                quicksum(z_group[jid, r, g] for g in allowed_groups[jid]) == active[jid, r],
                name=f'group_select_{jid}_{r}'
            )
            model.addConstr(
                quicksum(qty_group[jid, r, g] for g in allowed_groups[jid]) == qty[jid, r],
                name=f'qty_group_sum_{jid}_{r}'
            )
            for g in allowed_groups[jid]:
                model.addConstr(qty_group[jid, r, g] <= Q * z_group[jid, r, g],
                                name=f'qg_ub1_{jid}{r}{g}')
                model.addConstr(qty_group[jid, r, g] <= qty[jid, r],
                                name=f'qg_ub2_{jid}{r}{g}')
                model.addConstr(
                    qty_group[jid, r, g] >= qty[jid, r] - Q * (1 - z_group[jid, r, g]),
                    name=f'qg_lb_{jid}{r}{g}'
                )

    for t in tasks:
        jid, r, op = t
        model.addConstr(
            quicksum(x[jid, r, op, m] for m in eligible_machines_by_task[t]) == active[jid, r],
            name=f'assign_{jid}{r}{op}'
        )
        for m in eligible_machines_by_task[t]:
            g_m = data['machine_to_group'][m]
            if g_m in allowed_groups[jid]:
                model.addConstr(x[jid, r, op, m] <= z_group[jid, r, g_m],
                                name=f'machine_group_link_{jid}{r}{op}_{m}')
            else:
                model.addConstr(x[jid, r, op, m] == 0,
                                name=f'ineligible_machine_{jid}{r}{op}_{m}')

    for t in tasks:
        jid, r, op = t
        job    = job_by_id[jid]
        p_expr = quicksum(
            get_unit_time(data, job.part_no, op, g) * qty_group[jid, r, g]
            for g in allowed_groups[jid]
        )
        model.addConstr(proc_time[t] == p_expr, name=f'proc_time_def_{jid}{r}{op}')
        model.addConstr(C[t] == S[t] + proc_time[t], name=f'completion_{jid}{r}{op}')
        model.addConstr(S[t] <= M * active[jid, r], name=f'inactive_start_{jid}{r}{op}')
        model.addConstr(C[t] <= M * active[jid, r], name=f'inactive_finish_{jid}{r}{op}')

    for j in jobs:
        for r in split_ids:
            model.addConstr(
                S[j.job_id, r, 20] >= C[j.job_id, r, 10] - M * (1 - active[j.job_id, r]),
                name=f'tech_prec_{j.job_id}_{r}'
            )

    for j in jobs:
        due_min = dt_to_min(j.due_date, planning_start)
        for r in split_ids:
            model.addConstr(job_finish[j.job_id] >= C[j.job_id, r, 20],
                            name=f'job_finish_{j.job_id}_{r}')
        model.addConstr(
            job_finish[j.job_id] <= due_min + TARDINESS_LIMIT_MIN,
            name=f'tardiness_limit_{j.job_id}'
        )
        model.addConstr(makespan >= job_finish[j.job_id],
                        name=f'makespan_{j.job_id}')

    for m in machines:
        possible_tasks_m = [t for t in tasks if m in eligible_machines_by_task[t]]
        if not possible_tasks_m:
            continue

        model.addConstr(
            quicksum(first[t[0], t[1], t[2], m] for t in possible_tasks_m) <= 1,
            name=f'at_most_one_first_{m}'
        )
        model.addConstr(
            quicksum(last[t[0], t[1], t[2], m] for t in possible_tasks_m) <= 1,
            name=f'at_most_one_last_{m}'
        )

        for t in possible_tasks_m:
            tid = (t[0], t[1], t[2], m)
            pred_arcs = []
            succ_arcs = []
            for u in possible_tasks_m:
                if u == t:
                    continue
                k_pred = (u[0], u[1], u[2], t[0], t[1], t[2], m)
                k_succ = (t[0], t[1], t[2], u[0], u[1], u[2], m)
                if k_pred in arc:
                    pred_arcs.append(arc[k_pred])
                if k_succ in arc:
                    succ_arcs.append(arc[k_succ])

            model.addConstr(quicksum(pred_arcs) + first[tid] == x[tid],
                            name=f'pred_or_first_{t}_{m}')
            model.addConstr(quicksum(succ_arcs) + last[tid]  == x[tid],
                            name=f'succ_or_last_{t}_{m}')
            model.addConstr(first[tid] <= x[tid], name=f'first_link_{t}_{m}')
            model.addConstr(last[tid]  <= x[tid], name=f'last_link_{t}_{m}')
            model.addConstr(
                S[t] >= INITIAL_SETUP_MIN - M * (1 - first[tid]),
                name=f'initial_setup_time_{t}_{m}'
            )

    for key in arc_keys:
        i_jid, i_r, i_op, j_jid, j_r, j_op, m = key
        ti = (i_jid, i_r, i_op)
        tj = (j_jid, j_r, j_op)
        xi = x[i_jid, i_r, i_op, m]
        xj = x[j_jid, j_r, j_op, m]
        model.addConstr(arc[key] <= xi, name=f'arc_link_i_{key}')
        model.addConstr(arc[key] <= xj, name=f'arc_link_j_{key}')
        setup_ij = get_setup_time(data,
                                  job_by_id[i_jid].diameter,
                                  job_by_id[j_jid].diameter)
        model.addConstr(
            S[tj] >= C[ti] + setup_ij - M * (1 - arc[key]),
            name=f'global_no_overlap_setup_{key}'
        )

    for t in tasks:
        jid, r, op = t
        for w_idx, (l, u, _wname) in enumerate(overtime_windows):
            max_start  = model.addVar(lb=-M, ub=M, name=f'max_start_{jid}{r}{op}_{w_idx}')
            min_finish = model.addVar(lb=-M, ub=M, name=f'min_finish_{jid}{r}{op}_{w_idx}')
            diff       = model.addVar(lb=-M, ub=M, name=f'overlap_diff_{jid}{r}{op}_{w_idx}')
            model.addGenConstrMax(max_start,  [S[t]], constant=float(l),
                                  name=f'gc_max_start_{jid}{r}{op}_{w_idx}')
            model.addGenConstrMin(min_finish, [C[t]], constant=float(u),
                                  name=f'gc_min_finish_{jid}{r}{op}_{w_idx}')
            model.addConstr(diff == min_finish - max_start,
                            name=f'overlap_diff_def_{jid}{r}{op}_{w_idx}')
            model.addGenConstrMax(ot_vars[(t, w_idx)], [diff], constant=0.0,
                                  name=f'gc_overlap_pos_{jid}{r}{op}_{w_idx}')
            model.addConstr(ot_vars[(t, w_idx)] <= (u - l) * active[jid, r],
                            name=f'ot_active_ub_{jid}{r}{op}_{w_idx}')

            if DISALLOW_START_IN_OVERTIME:
                b = start_side[(t, w_idx)]
                model.addConstr(
                    S[t] <= float(l) + M * b + M * (1 - active[jid, r]),
                    name=f'start_before_1_{jid}{r}{op}_{w_idx}'
                )
                model.addConstr(
                    S[t] >= float(u) - M * (1 - b) - M * (1 - active[jid, r]),
                    name=f'start_before_2_{jid}{r}{op}_{w_idx}'
                )

    total_overtime = quicksum(
        ot_vars[(t, w_idx)]
        for t in tasks
        for w_idx in range(len(overtime_windows))
    )

    model.setObjective(total_overtime, GRB.MINIMIZE)
    model.update()

    n_binary = sum(1 for v in model.getVars() if v.VType == GRB.BINARY)
    n_int    = sum(1 for v in model.getVars() if v.VType == GRB.INTEGER)
    n_cont   = sum(1 for v in model.getVars() if v.VType == GRB.CONTINUOUS)

    print('\nModel buyuklugu:')
    print(f'  Toplam degisken : {model.NumVars:,}')
    print(f'  Binary          : {n_binary:,}')
    print(f'  Integer         : {n_int:,}')
    print(f'  Continuous      : {n_cont:,}')
    print(f'  Kisitlar        : {model.NumConstrs:,}')
    print(f'  GenKisitlar     : {model.NumGenConstrs:,}')
    print(f'\nGurobi log : {GUROBI_LOG_FILE}')
    print(f'Time limit : {TIME_LIMIT_SEC} sec')

    model.optimize()

    sol_count = model.SolCount
    print(f'\nGurobi Status  : {model.Status}')
    print(f'Solution Count : {sol_count}')

    if sol_count > 0:
        print(f'Objective      : {model.ObjVal:.4f}')
        print(f'Best Bound     : {model.ObjBound:.4f}')
        if model.IsMIP:
            print(f'MIP Gap        : {model.MIPGap * 100:.2f}%')
    else:
        print('UYARI: Time limit icinde feasible cozum bulunamadi (SolCount=0).')

    if model.Status == GRB.INFEASIBLE:
        print('Model infeasible -> IIS hesaplaniyor...')
        model.computeIIS()
        iis_path = BASE_DIR / 'model_iis.ilp'
        model.write(str(iis_path))
        print('IIS yazildi:', iis_path)
        return {
            'model': model, 'sol_count': 0,
            'objective_total_overtime': None,
            'objective_makespan': None,
        }

    if model.Status not in [GRB.OPTIMAL, GRB.TIME_LIMIT, GRB.SUBOPTIMAL] and sol_count == 0:
        print(f'UYARI: Beklenmeyen Gurobi status={model.Status}, SolCount=0.')
        return {
            'model': model, 'sol_count': 0,
            'objective_total_overtime': None,
            'objective_makespan': None,
        }

    return {
        'model':                     model,
        'sol_count':                 sol_count,
        'active':                    active,
        'qty':                       qty,
        'z_group':                   z_group,
        'qty_group':                 qty_group,
        'x':                         x,
        'S':                         S,
        'C':                         C,
        'proc_time':                 proc_time,
        'arc':                       arc,
        'first':                     first,
        'last':                      last,
        'job_finish':                job_finish,
        'makespan':                  makespan,
        'ot_vars':                   ot_vars,
        'overtime_windows':          overtime_windows,
        'tasks':                     tasks,
        'machines':                  machines,
        'split_ids':                 split_ids,
        'allowed_groups':            allowed_groups,
        'eligible_machines_by_task': eligible_machines_by_task,
        'job_by_id':                 job_by_id,
        'objective_total_overtime':  total_overtime.getValue() if sol_count > 0 else None,
        'objective_makespan':        makespan.X if sol_count > 0 else None,
    }


def value(v):
    try:
        return float(v.X)
    except Exception:
        return float(v)


def extract_solution_tables(data, sol):
    if sol.get('sol_count', 0) == 0:
        print('Cozum yok, schedule bos donduruluyor.')
        return pd.DataFrame(), pd.DataFrame()

    jobs           = data['jobs']
    job_by_id      = sol['job_by_id']
    split_ids      = sol['split_ids']
    tasks          = sol['tasks']
    active         = sol['active']
    qty            = sol['qty']
    z_group        = sol['z_group']
    x              = sol['x']
    S              = sol['S']
    C              = sol['C']
    proc_time      = sol['proc_time']
    arc            = sol['arc']
    first          = sol['first']
    planning_start = data['planning_start']

    split_rows     = []
    selected_qty   = {}
    selected_group = {}

    for j in jobs:
        for r in split_ids:
            a = value(active[j.job_id, r])
            q = round(value(qty[j.job_id, r]))
            if a > 0.5:
                g_sel = None
                for g in sol['allowed_groups'][j.job_id]:
                    if value(z_group[j.job_id, r, g]) > 0.5:
                        g_sel = g
                        break
                selected_qty[(j.job_id, r)]   = int(q)
                selected_group[(j.job_id, r)] = g_sel
                split_rows.append({
                    'job_id':            j.job_id,
                    'part_no':           j.part_no,
                    'split_id':          r,
                    'quantity':          int(q),
                    'selected_group':    g_sel,
                    'original_quantity': j.quantity,
                    'due_date':          j.due_date,
                })

    selected_machine = {}
    for t in tasks:
        jid, r, op = t
        if value(active[jid, r]) <= 0.5:
            continue
        for m in sol['eligible_machines_by_task'][t]:
            if (jid, r, op, m) in x and value(x[jid, r, op, m]) > 0.5:
                selected_machine[t] = m
                break

    pred_task     = {}
    setup_by_task = {}
    first_flag    = {}

    for t in tasks:
        jid, r, op = t
        if value(active[jid, r]) <= 0.5:
            continue
        m = selected_machine[t]
        first_flag[t] = value(first[jid, r, op, m]) > 0.5
        if first_flag[t]:
            pred_task[t]     = None
            setup_by_task[t] = INITIAL_SETUP_MIN
        else:
            pred  = None
            setup = 0.0
            for key, var in arc.items():
                i_jid, i_r, i_op, j_jid, j_r, j_op, mm = key
                if mm == m and (j_jid, j_r, j_op) == t and value(var) > 0.5:
                    pred  = (i_jid, i_r, i_op)
                    setup = get_setup_time(data,
                                           job_by_id[i_jid].diameter,
                                           job_by_id[j_jid].diameter)
                    break
            pred_task[t]     = pred
            setup_by_task[t] = setup

    rows = []
    for t in tasks:
        jid, r, op = t
        if value(active[jid, r]) <= 0.5:
            continue
        job             = job_by_id[jid]
        m               = selected_machine[t]
        start_min       = value(S[t])
        finish_min      = value(C[t])
        process_min     = value(proc_time[t])
        setup_min       = setup_by_task[t]
        setup_start_min = max(0.0, start_min - setup_min)
        pred            = pred_task[t]
        ot_min          = overtime_overlap_minutes(
                              min_to_dt(start_min,  planning_start),
                              min_to_dt(finish_min, planning_start)
                          )
        rows.append({
            'job_id':               jid,
            'part_no':              job.part_no,
            'split_id':             r,
            'split_count':          sum(1 for rr in split_ids
                                        if value(active[jid, rr]) > 0.5),
            'quantity':             selected_qty[(jid, r)],
            'diameter':             job.diameter,
            'group':                selected_group[(jid, r)],
            'operation':            op,
            'machine':              m,
            'start_min':            start_min,
            'finish_min':           finish_min,
            'setup_start_min':      setup_start_min,
            'setup_finish_min':     start_min,
            'processing_min':       process_min,
            'setup_min':            setup_min,
            'overtime_min':         ot_min,
            'start':                min_to_dt(start_min,        planning_start),
            'finish':               min_to_dt(finish_min,       planning_start),
            'setup_start':          min_to_dt(setup_start_min,  planning_start),
            'setup_finish':         min_to_dt(start_min,        planning_start),
            'prev_task': ('SRC' if pred is None
                          else f"{job_by_id[pred[0]].part_no} S{pred[1]} Op{pred[2]}"),
            'is_first_on_machine': pred is None,
        })

    schedule_df = pd.DataFrame(rows)
    if not schedule_df.empty:
        schedule_df = (schedule_df
                       .sort_values(['machine', 'start_min', 'finish_min'])
                       .reset_index(drop=True))
        schedule_df['global_sequence_on_machine'] = (
            schedule_df.groupby('machine').cumcount() + 1
        )

    return schedule_df, pd.DataFrame(split_rows)


def check_machine_global_nooverlap(schedule_df, data, tolerance_min=1e-5):
    rows = []
    df   = schedule_df.copy().sort_values(['machine', 'start_min', 'finish_min'])
    for machine, g in df.groupby('machine'):
        recs = g.to_dict('records')
        for i in range(len(recs) - 1):
            cur = recs[i]
            nxt = recs[i + 1]
            req     = (cur['finish_min']
                       + get_setup_time(data, cur['diameter'], nxt['diameter']))
            slack   = nxt['start_min'] - req
            overlap = cur['finish_min'] - nxt['start_min']
            rows.append({
                'machine':                       machine,
                'current_task':                  f"{cur['part_no']} S{cur['split_id']} Op{cur['operation']}",
                'next_task':                     f"{nxt['part_no']} S{nxt['split_id']} Op{nxt['operation']}",
                'current_finish_min':            cur['finish_min'],
                'required_next_start_after_setup': req,
                'next_start_min':                nxt['start_min'],
                'setup_between_min':             get_setup_time(data, cur['diameter'], nxt['diameter']),
                'slack_min':                     slack,
                'processing_overlap_min':        max(0.0, overlap),
                'Feasible':                      slack >= -tolerance_min,
            })
    if not rows:
        return pd.DataFrame([{'status': 'NO PAIRS TO CHECK', 'Feasible': True}])
    return pd.DataFrame(rows)


def build_validation_tables(data, sol, schedule_df, split_df):
    if schedule_df.empty:
        return {'Feasibility_Summary': pd.DataFrame([{'Check': 'No solution', 'All_Feasible': False, 'Violations': 0}])}

    jobs      = data['jobs']
    split_ids = sol['split_ids']
    tasks     = sol['tasks']
    active    = sol['active']
    qty       = sol['qty']
    S         = sol['S']
    C         = sol['C']
    proc_time = sol['proc_time']
    x         = sol['x']
    job_by_id = sol['job_by_id']

    assignment_rows = []
    completion_rows = []
    precedence_rows = []
    tardiness_rows  = []
    quantity_rows   = []
    machine_nooverlap_df = check_machine_global_nooverlap(schedule_df, data)

    for j in jobs:
        total_q = sum(round(value(qty[j.job_id, r])) for r in split_ids)
        quantity_rows.append({
            'job_id':             j.job_id,
            'part_no':            j.part_no,
            'original_quantity':  j.quantity,
            'sum_split_quantity': total_q,
            'Feasible':           abs(total_q - j.quantity) <= EPS,
        })
        finish_vals = [value(C[j.job_id, r, 20])
                       for r in split_ids
                       if value(active[j.job_id, r]) > 0.5]
        job_fin = max(finish_vals) if finish_vals else 0.0
        due_min = dt_to_min(j.due_date, data['planning_start'])
        tard    = max(0.0, job_fin - due_min)
        viol    = max(0.0, job_fin - (due_min + TARDINESS_LIMIT_MIN))
        tardiness_rows.append({
            'job_id':                        j.job_id,
            'part_no':                       j.part_no,
            'job_finish_min':                job_fin,
            'due_min':                       due_min,
            'tardiness_min_from_due':        tard,
            'tardiness_limit_min':           TARDINESS_LIMIT_MIN,
            'tardiness_limit_violation_min': viol,
            'Feasible':                      viol <= EPS,
        })

    for t in tasks:
        jid, r, op = t
        a          = value(active[jid, r])
        assign_sum = sum(value(x[jid, r, op, m])
                         for m in sol['eligible_machines_by_task'][t]
                         if (jid, r, op, m) in x)
        assignment_rows.append({
            'job_id': jid, 'split_id': r, 'operation': op,
            'active': a, 'assignment_sum': assign_sum,
            'Feasible': abs(assign_sum - a) <= EPS,
        })
        lhs = value(C[t])
        rhs = value(S[t]) + value(proc_time[t])
        completion_rows.append({
            'job_id': jid, 'split_id': r, 'operation': op,
            'C': lhs, 'S_plus_processing': rhs,
            'Deviation': abs(lhs - rhs),
            'Feasible':  abs(lhs - rhs) <= EPS,
        })

    for j in jobs:
        for r in split_ids:
            if value(active[j.job_id, r]) > 0.5:
                slack = value(S[j.job_id, r, 20]) - value(C[j.job_id, r, 10])
                precedence_rows.append({
                    'job_id':          j.job_id,
                    'split_id':        r,
                    'start_op20_min':  value(S[j.job_id, r, 20]),
                    'finish_op10_min': value(C[j.job_id, r, 10]),
                    'slack_min':       slack,
                    'Feasible':        slack >= -EPS,
                })

    dfs = {
        'Split_Quantity_Check':     pd.DataFrame(quantity_rows),
        'Assignment_Check':         pd.DataFrame(assignment_rows),
        'Completion_Check':         pd.DataFrame(completion_rows),
        'Precedence_Check':         pd.DataFrame(precedence_rows),
        'Tardiness_Check':          pd.DataFrame(tardiness_rows),
        'Machine_Global_NoOverlap': machine_nooverlap_df,
    }

    summary = []
    for name, df in dfs.items():
        if 'Feasible' in df.columns:
            summary.append({
                'Check':        name,
                'All_Feasible': bool(df['Feasible'].all()),
                'Violations':   int((~df['Feasible']).sum()),
            })
    dfs['Feasibility_Summary'] = pd.DataFrame(summary)
    return dfs


def build_machine_summary(schedule_df):
    return (schedule_df.groupby('machine')
            .agg(
                total_processing_min=('processing_min', 'sum'),
                total_setup_min=('setup_min', 'sum'),
                total_overtime_min=('overtime_min', 'sum'),
                first_start=('start', 'min'),
                last_finish=('finish', 'max'),
                task_count=('job_id', 'count'),
            )
            .reset_index()
            .sort_values('total_overtime_min', ascending=False))


def build_job_summary(schedule_df, data):
    jobs_by_id = {j.job_id: j for j in data['jobs']}
    comp       = (schedule_df[schedule_df['operation'] == 20]
                  .groupby('job_id')['finish'].max().reset_index())
    rows = []
    for _, r in comp.iterrows():
        job      = jobs_by_id[r['job_id']]
        finish   = r['finish']
        tardiness= max(0.0, (finish - job.due_date).total_seconds() / 60.0)
        rows.append({
            'job_id':                       job.job_id,
            'part_no':                      job.part_no,
            'quantity':                     job.quantity,
            'due_date':                     job.due_date,
            'completion':                   finish,
            'tardiness_min_from_due':       tardiness,
            'tardiness_days_from_due':      tardiness / 1440.0,
            'allowed_latest_completion':    job.due_date + timedelta(minutes=TARDINESS_LIMIT_MIN),
            'tardiness_limit_violation_min': max(
                0.0,
                (finish - (job.due_date + timedelta(minutes=TARDINESS_LIMIT_MIN))).total_seconds() / 60.0
            ),
        })
    return pd.DataFrame(rows).sort_values(['due_date', 'part_no'])


def build_split_summary(schedule_df, data):
    jobs_by_id = {j.job_id: j for j in data['jobs']}
    rows = []
    for job_id, g in schedule_df.groupby('job_id'):
        job = jobs_by_id[job_id]
        sq  = (g[['split_id', 'quantity']].drop_duplicates()
                .sort_values('split_id')
                .apply(lambda r: f"S{int(r['split_id'])}={int(r['quantity'])}", axis=1)
                .tolist())
        rows.append({
            'job_id':            job_id,
            'part_no':           job.part_no,
            'original_quantity': job.quantity,
            'split_count':       int(g['split_count'].max()),
            'is_split':          'YES' if int(g['split_count'].max()) > 1 else 'NO',
            'split_quantities':  ', '.join(sq),
            'due_date':          job.due_date,
            'final_completion':  g[g['operation'] == 20]['finish'].max(),
        })
    return pd.DataFrame(rows).sort_values(
        ['is_split', 'due_date', 'part_no'], ascending=[False, True, True]
    )


def export_excel_outputs(data, sol, schedule_df, split_df, validation_dfs):
    if sol.get('sol_count', 0) == 0 or schedule_df.empty:
        metrics_only = pd.DataFrame([{
            'gurobi_status':  sol['model'].Status,
            'sol_count':      sol.get('sol_count', 0),
            'best_bound':     getattr(sol['model'], 'ObjBound', None),
            'baseline_overtime_min': data.get('baseline_overtime_min', 0.0),
            'big_m_used':     BIG_M,
            'tardiness_limit_days': TARDINESS_LIMIT_DAYS,
            'gurobi_log':     str(GUROBI_LOG_FILE),
        }])
        with pd.ExcelWriter(OUTPUT_SCHEDULE_FILE, engine='openpyxl') as writer:
            metrics_only.to_excel(writer, sheet_name='Final Metrics', index=False)
        print(f'Cozum bulunamadi, sadece metrikler kaydedildi: {OUTPUT_SCHEDULE_FILE}')
        return

    machine_summary = build_machine_summary(schedule_df)
    job_summary     = build_job_summary(schedule_df, data)
    split_summary   = build_split_summary(schedule_df, data)

    metrics = pd.DataFrame([{
        'objective_value':               sol['model'].ObjVal,
        'total_processing_overtime_min': schedule_df['overtime_min'].sum(),
        'baseline_overtime_min':         data.get('baseline_overtime_min', 0.0),
        'overtime_improvement_min':      data.get('baseline_overtime_min', 0.0)
                                          - schedule_df['overtime_min'].sum(),
        'total_setup_min':               schedule_df['setup_min'].sum(),
        'makespan_min':                  sol['objective_makespan'],
        'gurobi_status':                 sol['model'].Status,
        'mip_gap':                       getattr(sol['model'], 'MIPGap', None),
        'big_m_used':                    BIG_M,
        'tardiness_limit_days':          TARDINESS_LIMIT_DAYS,
        'gurobi_log':                    str(GUROBI_LOG_FILE),
    }])

    sout = schedule_df.copy()
    sout['Tarih']                      = pd.to_datetime(sout['start']).dt.date
    sout['Başlangıç Saat']             = pd.to_datetime(sout['start']).dt.strftime('%H:%M')
    sout['Bitiş Saat']                 = pd.to_datetime(sout['finish']).dt.strftime('%H:%M')
    sout['Parça No']                   = sout['part_no']
    sout['Makine No']                  = sout['machine']
    sout['CNC-1 operasyonu(piston)']   = np.where(sout['operation'] == 10, 'X', '')
    sout['CNC-2 operasyonu (saplama)'] = np.where(sout['operation'] == 20, 'X', '')
    sout['Adet']                       = sout['quantity']
    sout['Çap']                        = sout['diameter']
    sout['Makine Grubu']               = sout['group']
    sout['Fazla mesai']                = sout['overtime_min'].round(1)
    sout['Split']                      = sout['split_id']
    sout['Split Count']                = sout['split_count']
    sout['Split Label']                = 'S' + sout['split_id'].astype(str) + '/' + sout['split_count'].astype(str)
    sout['Operation Label']            = np.where(sout['operation'] == 10, 'Op10', 'Op20')
    sout['Setup Süresi']               = sout['setup_min'].round(1)
    sout['İşlem Süresi']               = sout['processing_min'].round(1)
    sout['Job ID']                     = sout['job_id']

    with pd.ExcelWriter(OUTPUT_SCHEDULE_FILE, engine='openpyxl') as writer:
        sout.to_excel(writer, sheet_name='Optimized Schedule', index=False)
        machine_summary.to_excel(writer, sheet_name='Machine Summary', index=False)
        job_summary.to_excel(writer, sheet_name='Job Summary', index=False)
        split_summary.to_excel(writer, sheet_name='Split Summary', index=False)
        metrics.to_excel(writer, sheet_name='Final Metrics', index=False)

    with pd.ExcelWriter(OUTPUT_VALIDATION_FILE, engine='openpyxl') as writer:
        validation_dfs['Feasibility_Summary'].to_excel(
            writer, sheet_name='Feasibility_Summary', index=False)
        for name, df in validation_dfs.items():
            if name == 'Feasibility_Summary':
                continue
            df.to_excel(writer, sheet_name=name[:31], index=False)


def export_gantt_html(schedule_df, output_html_path):
    if schedule_df.empty:
        with open(output_html_path, 'w', encoding='utf-8') as f:
            f.write('<html><body><h2>Schedule bos - feasible cozum bulunamadi.</h2></body></html>')
        return

    records = []
    palette = {
        'Op.10':       '#3b82f6',
        'Op.10 Split': '#f59e0b',
        'Op.20':       '#10b981',
        'Op.20 Split': '#f97316',
        'Setup':       '#111111',
    }

    for _, task in schedule_df.iterrows():
        split_flag = int(task['split_count']) > 1
        legend = ('Op.10 Split' if (task['operation'] == 10 and split_flag)
                  else 'Op.10'  if task['operation'] == 10
                  else 'Op.20 Split' if split_flag
                  else 'Op.20')
        label = (f"{task['part_no']} S{int(task['split_id'])}/"
                 f"{int(task['split_count'])} | Op{int(task['operation'])}")

        if task['setup_min'] > EPS:
            records.append({
                'Legend': 'Setup', 'Machine': task['machine'],
                'Start_min': task['setup_start_min'],
                'Finish_min': task['setup_finish_min'],
                'Duration_min': task['setup_min'],
                'Text': '', 'Task': f'SETUP -> {label}',
                'Job': task['job_id'], 'Part': task['part_no'],
                'Split': task['split_id'], 'Operation': task['operation'],
                'Qty': task['quantity'], 'Setup_min': task['setup_min'],
                'Processing_min': 0,
                'Start_time': str(task['setup_start']),
                'Finish_time': str(task['setup_finish']),
            })

        records.append({
            'Legend': legend, 'Machine': task['machine'],
            'Start_min': task['start_min'], 'Finish_min': task['finish_min'],
            'Duration_min': task['processing_min'],
            'Text': label, 'Task': label,
            'Job': task['job_id'], 'Part': task['part_no'],
            'Split': task['split_id'], 'Operation': task['operation'],
            'Qty': task['quantity'], 'Setup_min': task['setup_min'],
            'Processing_min': task['processing_min'],
            'Start_time': str(task['start']),
            'Finish_time': str(task['finish']),
        })

    gantt_df      = (pd.DataFrame(records)
                     .sort_values(['Machine', 'Start_min', 'Finish_min'])
                     .reset_index(drop=True))
    machine_order = sorted(gantt_df['Machine'].unique().tolist())
    legend_order  = ['Op.10', 'Op.10 Split', 'Op.20', 'Op.20 Split', 'Setup']
    makespan      = schedule_df['finish_min'].max()

    fig = go.Figure()
    for lg in legend_order:
        sub = gantt_df[gantt_df['Legend'] == lg].copy()
        if sub.empty:
            continue
        fig.add_trace(go.Bar(
            x=sub['Duration_min'].tolist(),
            y=sub['Machine'].tolist(),
            base=sub['Start_min'].tolist(),
            orientation='h',
            name=lg,
            marker=dict(
                color=palette[lg],
                line=dict(
                    color='#111111' if 'Split' in lg else '#ffffff',
                    width=1.4 if 'Split' in lg else 0.4
                ),
                pattern=dict(shape='/' if 'Split' in lg else ''),
            ),
            text=sub['Text'].tolist(),
            textposition='inside',
            insidetextanchor='middle',
            textfont=dict(color='white', size=10),
            customdata=list(zip(
                sub['Job'], sub['Part'], sub['Split'], sub['Operation'],
                sub['Qty'], sub['Setup_min'], sub['Processing_min'],
                sub['Start_min'], sub['Finish_min'],
                sub['Start_time'], sub['Finish_time'], sub['Task']
            )),
            hovertemplate=(
                'Makine: %{y}<br>Task: %{customdata[11]}<br>'
                'Job: %{customdata[0]}<br>Part: %{customdata[1]}<br>'
                'Split: %{customdata[2]}<br>Operation: %{customdata[3]}<br>'
                'Qty: %{customdata[4]}<br>Setup min: %{customdata[5]:.2f}<br>'
                'Processing min: %{customdata[6]:.2f}<br>'
                'Start min: %{customdata[7]:.2f}<br>'
                'Finish min: %{customdata[8]:.2f}<br>'
                'Start time: %{customdata[9]}<br>'
                'Finish time: %{customdata[10]}<extra></extra>'
            )
        ))

    fig.add_shape(
        type='line', x0=makespan, x1=makespan, y0=0, y1=1,
        xref='x', yref='paper',
        line=dict(width=2, dash='dash', color='#dc2626'),
    )
    fig.add_annotation(
        x=makespan, y=1.02, xref='x', yref='paper',
        text=f'Cmax={makespan:.0f} dk',
        showarrow=False, xanchor='left',
        font=dict(color='#dc2626', size=11),
        bgcolor='rgba(255,255,255,0.7)',
    )
    fig.update_layout(
        barmode='overlay',
        title='Exact Gurobi Schedule Gantt (Small)<br>'
              '<sup>Setup=siyah, Op10=mavi, Op20=yesil, split=desenli</sup>',
        xaxis_title='Zaman (dk)',
        yaxis_title='Makine',
        yaxis=dict(categoryorder='array',
                   categoryarray=machine_order,
                   autorange='reversed'),
        legend_title='Gorev Turu',
        plot_bgcolor='white', paper_bgcolor='white',
        hovermode='closest',
        width=2500,
        height=max(650, 70 * len(machine_order)),
        margin=dict(l=80, r=40, t=110, b=60),
    )
    fig.update_xaxes(showgrid=True, gridcolor='#e5e7eb', zeroline=False)
    fig.update_yaxes(showgrid=True, gridcolor='#f3f4f6', zeroline=False)

    n = len(fig.data)
    def vis(names):
        return [fig.data[i].name in names for i in range(n)]

    fig.update_layout(updatemenus=[dict(
        type='buttons', direction='right', x=0, y=1.18, showactive=True,
        buttons=[
            dict(label='Tümü',               method='update', args=[{'visible': [True] * n}]),
            dict(label='Sadece Operasyonlar', method='update',
                 args=[{'visible': vis({'Op.10', 'Op.10 Split', 'Op.20', 'Op.20 Split'})}]),
            dict(label='Sadece Op.10',        method='update',
                 args=[{'visible': vis({'Op.10', 'Op.10 Split'})}]),
            dict(label='Sadece Op.20',        method='update',
                 args=[{'visible': vis({'Op.20', 'Op.20 Split'})}]),
            dict(label='Sadece Setup',        method='update',
                 args=[{'visible': vis({'Setup'})}]),
            dict(label='Setup Hariç',         method='update',
                 args=[{'visible': vis({'Op.10', 'Op.10 Split', 'Op.20', 'Op.20 Split'})}]),
        ],
    )])

    fig.write_html(str(output_html_path), include_plotlyjs='cdn', full_html=True)


if __name__ == '__main__':
    print('\nVeri yukleniyor...')
    GLOBAL_DATA = build_problem_data(
        shipment_file=SHIPMENT_FILE,
        sdst_file=SDST_FILE,
        machine_group_file=MACHINE_GROUP_FILE,
        use_shipped_quantity=True,
    )

    print(f"Job sayisi    : {len(GLOBAL_DATA['jobs'])}")
    print(f"Makineler     : {sorted(GLOBAL_DATA['machine_to_group'].keys())}")
    print(f"Planlama bas  : {GLOBAL_DATA['planning_start']}")
    print(f"Gecikme limiti: {TARDINESS_LIMIT_DAYS} gun")
    print(f"Baseline OT   : {GLOBAL_DATA['baseline_overtime_min']:.0f} dk")

    print('\nGurobi MILP kuruluyor ve cozuluyor...')
    solution = solve_exact_gurobi(GLOBAL_DATA)

    print('\nCozum cikariliyor...')
    schedule_df, split_df = extract_solution_tables(GLOBAL_DATA, solution)
    validation_dfs = build_validation_tables(GLOBAL_DATA, solution, schedule_df, split_df)

    print('Excel ciktilari olusturuluyor...')
    export_excel_outputs(GLOBAL_DATA, solution, schedule_df, split_df, validation_dfs)

    print('HTML Gantt olusturuluyor...')
    export_gantt_html(schedule_df, OUTPUT_GANTT_FILE)

    print('\nTamamlandi.')
    print('Schedule Excel   :', OUTPUT_SCHEDULE_FILE)
    print('Validation Excel :', OUTPUT_VALIDATION_FILE)
    print('Gantt HTML       :', OUTPUT_GANTT_FILE)
    print('Gurobi log       :', GUROBI_LOG_FILE)
    if solution.get('sol_count', 0) > 0:
        print('Objective (OT dk):', solution['model'].ObjVal)
        print('Toplam OT (dk)   :', schedule_df['overtime_min'].sum())
        print('Baseline OT (dk) :', GLOBAL_DATA['baseline_overtime_min'])
        print('Iyilesme (dk)    :',
              GLOBAL_DATA['baseline_overtime_min'] - schedule_df['overtime_min'].sum())
    else:
        print('Feasible cozum bulunamadi.')


# ============================================================
# EXACT MILP - CPU TIME AND PEAK MEMORY
# ============================================================

import os
import sys
import threading
import subprocess
import pandas as pd

from datetime import datetime, timedelta, time
import time as time_module

try:
    import psutil
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "psutil"])
    import psutil


def get_total_memory_mb(root_process):
    total_bytes = 0
    process_list = [root_process]
    try:
        process_list.extend(root_process.children(recursive=True))
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        pass
    for process_item in process_list:
        try:
            total_bytes += process_item.memory_info().rss
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    return total_bytes / (1024 ** 2)


def get_total_cpu_seconds(root_process):
    total_cpu_seconds = 0.0
    process_list = [root_process]
    try:
        process_list.extend(root_process.children(recursive=True))
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        pass
    for process_item in process_list:
        try:
            cpu_times = process_item.cpu_times()
            total_cpu_seconds += (cpu_times.user + cpu_times.system)
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    return total_cpu_seconds


current_process = psutil.Process(os.getpid())
baseline_memory_mb = get_total_memory_mb(current_process)
peak_memory_mb = baseline_memory_mb
start_cpu_seconds = get_total_cpu_seconds(current_process)
start_wall_clock = time_module.perf_counter()
stop_monitor = threading.Event()


def monitor_memory():
    global peak_memory_mb
    while not stop_monitor.is_set():
        current_memory_mb = get_total_memory_mb(current_process)
        if current_memory_mb > peak_memory_mb:
            peak_memory_mb = current_memory_mb
        time_module.sleep(0.05)


monitor_thread = threading.Thread(target=monitor_memory, daemon=True)
monitor_thread.start()

print("\nExact MILP yeniden calistiriliyor...")
print("CPU time ve peak memory olculuyor.")

try:
    measured_exact_solution = solve_exact_gurobi(GLOBAL_DATA)
finally:
    wall_clock_seconds = time_module.perf_counter() - start_wall_clock
    stop_monitor.set()
    monitor_thread.join(timeout=2.0)
    peak_memory_mb = max(peak_memory_mb, get_total_memory_mb(current_process))
    end_cpu_seconds = get_total_cpu_seconds(current_process)
    cpu_time_seconds = max(0.0, end_cpu_seconds - start_cpu_seconds)

additional_peak_memory_mb = max(0.0, peak_memory_mb - baseline_memory_mb)

print("\nFonksiyonun donduregu veri tipi:")
print(type(measured_exact_solution))

if isinstance(measured_exact_solution, dict):
    measured_model = measured_exact_solution.get("model")
elif isinstance(measured_exact_solution, tuple):
    measured_model = None
    for returned_item in measured_exact_solution:
        if hasattr(returned_item, "optimize") and hasattr(returned_item, "Status"):
            measured_model = returned_item
            break
elif hasattr(measured_exact_solution, "optimize") and hasattr(measured_exact_solution, "Status"):
    measured_model = measured_exact_solution
else:
    measured_model = None

if measured_model is not None:
    status_names = {
        2: "OPTIMAL", 3: "INFEASIBLE", 4: "INF_OR_UNBD", 5: "UNBOUNDED",
        9: "TIME_LIMIT", 11: "INTERRUPTED", 13: "SUBOPTIMAL"
    }
    solver_status = status_names.get(measured_model.Status, str(measured_model.Status))
    objective_value = float(measured_model.ObjVal) if measured_model.SolCount > 0 else None
    mip_gap_percent = (
        float(measured_model.MIPGap) * 100
        if measured_model.SolCount > 0 and measured_model.IsMIP else None
    )
else:
    solver_status = "Model object was not returned"
    objective_value = None
    mip_gap_percent = None
    if isinstance(measured_exact_solution, dict):
        for possible_key in ["objective", "objective_value", "total_overtime",
                             "total_overtime_min", "obj_val"]:
            if possible_key in measured_exact_solution:
                objective_value = measured_exact_solution[possible_key]
                break

exact_resource_df = pd.DataFrame([{
    "Method": "Exact MILP (Small)",
    "Solver Status": solver_status,
    "Objective Value": objective_value,
    "MIP Gap (%)": mip_gap_percent,
    "Wall-Clock Time (s)": wall_clock_seconds,
    "CPU Time (s)": cpu_time_seconds,
    "Baseline Memory (MB)": baseline_memory_mb,
    "Peak Memory (MB)": peak_memory_mb,
    "Additional Peak Memory (MB)": additional_peak_memory_mb,
    "Big-M Used": BIG_M,
    "Gurobi Log": str(GUROBI_LOG_FILE),
}])

print("\n" + "=" * 75)
print("EXACT MILP (SMALL) - COMPUTATIONAL PERFORMANCE")
print("=" * 75)
print(f"Solver status           : {solver_status}")
print(f"Objective value         : {objective_value}")
print(f"MIP gap (%)             : {mip_gap_percent}")
print(f"Wall-clock time         : {wall_clock_seconds:.4f} seconds")
print(f"CPU time                : {cpu_time_seconds:.4f} seconds")
print(f"Baseline memory         : {baseline_memory_mb:.4f} MB")
print(f"Peak memory             : {peak_memory_mb:.4f} MB")
print(f"Additional peak memory  : {additional_peak_memory_mb:.4f} MB")
print("=" * 75)

display(exact_resource_df.round(4))

if "BASE_DIR" in globals():
    exact_performance_path = BASE_DIR / "exact_cpu_time_peak_memory_small.xlsx"
else:
    exact_performance_path = "exact_cpu_time_peak_memory_small.xlsx"

exact_resource_df.to_excel(exact_performance_path, index=False)

print("\nPerformance file saved to:")
print(exact_performance_path)

Gerekli Python paketleri hazır.
Shipment file         : C:\Users\emrec\Documents\parallel_scheduling\exactkucuksplit\simple_test_sevkiyat.xlsx
SDST file             : C:\Users\emrec\Documents\parallel_scheduling\exactkucuksplit\SDST.xlsx
Machine-group file    : C:\Users\emrec\Documents\parallel_scheduling\exactkucuksplit\machine_group_data_simple.xlsx

Using files:
BASE_DIR           = C:\Users\emrec\Documents\parallel_scheduling\exactkucuksplit
SHIPMENT_FILE      = C:\Users\emrec\Documents\parallel_scheduling\exactkucuksplit\simple_test_sevkiyat.xlsx
SDST_FILE          = C:\Users\emrec\Documents\parallel_scheduling\exactkucuksplit\SDST.xlsx
MACHINE_GROUP_FILE = C:\Users\emrec\Documents\parallel_scheduling\exactkucuksplit\machine_group_data_simple.xlsx
GUROBI_LOG_FILE    = C:\Users\emrec\Documents\parallel_scheduling\exactkucuksplit\gurobi_exact_small.log

Outputs will be saved in: C:\Users\emrec\Documents\parallel_scheduling\exactkucuksplit

Veri yukleniyor...
Job sayisi    : 3
Makine

,Method,Solver Status,Objective Value,MIP Gap (%),Wall-Clock Time (s),CPU Time (s),Baseline Memory (MB),Peak Memory (MB),Additional Peak Memory (MB),Big-M Used,Gurobi Log
0,Exact MILP (Small),OPTIMAL,0.0,0.0,0.1324,0.2344,169.0625,170.4883,1.4258,10000,C:\Users\emrec\Documents\parallel_scheduling\e...



Performance file saved to:
C:\Users\emrec\Documents\parallel_scheduling\exactkucuksplit\exact_cpu_time_peak_memory_small.xlsx
